In [1]:
import re
import uuid
from pyspark.sql import DataFrame
import pyspark.sql.functions as F

StatementMeta(, 67438dec-c9ba-46f6-9aeb-0c26e01b0816, 3, Finished, Available, Finished, False)

# LOAD TO CSV

In [ ]:
def param_or_default(*names, default=None):
    for name in names:
        value = globals().get(name)
        if value not in (None, ""):
            return value
    return default


def to_snake_case(string):
    string = re.sub(r'([a-z])([A-Z0-9])', r'\1_\2', string)
    string = re.sub(r'([A-Z])([A-Z][a-z])', r'\1_\2', string)
    string = string.lower().strip()
    return string


def trim_all_string_columns(df):
    string_columns = [
        field.name
        for field in df.schema.fields
        if field.dataType.typeName() == "string"
    ]

    return df.select(
        *[
            F.trim(F.col(c)).alias(c) if c in string_columns else F.col(c)
            for c in df.columns
        ]
    )


def replace_old_dates(df):
    date_columns = [
        field.name
        for field in df.schema.fields
        if field.dataType.typeName() in ("date", "timestamp")
    ]

    for date_column in date_columns:
        df = df.withColumn(
            date_column,
            F.when(
                F.col(date_column) <= "1900-01-01",
                F.to_date(F.lit("1900-01-01"), "yyyy-MM-dd")
            ).otherwise(F.col(date_column))
        )

    return df


def clean_bronze_table(df, limit_rows_for_debugging=False):
    if limit_rows_for_debugging:
        df = df.limit(1000)

    snake_case_column_names = [to_snake_case(col) for col in df.columns]
    df = df.toDF(*snake_case_column_names)
    df = replace_old_dates(df)
    df = trim_all_string_columns(df)

    return df


def add_metadata(
    df: DataFrame,
    ingest_date: str,
    file_path: str,
    source_system: str,
    table_name: str,
    lineage_id: str
) -> DataFrame:
    return (
        df
        .withColumn("_ingest_time", F.current_timestamp())
        .withColumn("_ingest_date", F.to_date(F.lit(ingest_date)))
        .withColumn("_source_file", F.lit(file_path))
        .withColumn("_source_system", F.lit(source_system))
        .withColumn("_table", F.lit(table_name))
        .withColumn("_lineage_id", F.lit(lineage_id))
    )


def remove_exact_duplicates(df):
    return df.dropDuplicates(df.columns)


def new_lineage_id():
    return str(uuid.uuid4())

# TRANSACTIONAL TO CURATED

In [1]:
from delta.tables import DeltaTable
from pyspark.sql.functions import *

def validate_duplicates(df, key_col="hashed_pk", limit=10):
    dup_keys_df = (
        df.groupBy(key_col)
          .agg(count("*").alias("duplicate_count"))
          .filter(col("duplicate_count") > 1)
          .orderBy(col("duplicate_count").desc(), col(key_col))
    )

    total_dup_groups = dup_keys_df.count()
    print(f"Total duplicate {key_col} groups: {total_dup_groups}")
    display(dup_keys_df.limit(limit))

    if total_dup_groups > 0:
        dup_records_df = (
            df.join(dup_keys_df.select(key_col), on=key_col, how="inner")
              .orderBy(key_col)
        )
        display(dup_records_df.limit(limit))



def merge_to_target(df, target_schema, target_table, location_path, merge_key="hashed_pk"):
    full_table_name = f"{target_schema}.{target_table}"

    if not DeltaTable.isDeltaTable(spark, location_path):
        (
            df.write
            .format("delta")
            .mode("overwrite")
            .option("path", location_path)
            .option("overwriteSchema", "true")
            .saveAsTable(full_table_name)
        )
        print(f"Created {full_table_name}")
    else:
        delta_table = DeltaTable.forPath(spark, location_path)

        (
            delta_table.alias("t")
            .merge(
                df.alias("s"),
                f"t.{merge_key} = s.{merge_key}"
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )

        print(f"Merged into {full_table_name}")

StatementMeta(, cb966635-4bfd-4d9c-9ed8-6c748b036df2, 3, Finished, Available, Finished, False)